# Extract Comments & Track Changes from .docx File

Extracts comments and track changes (insertions, deletions, formatting) from:
`docs/KKP/LNC/REVISEDMutual Nondisclosure Agreement (V2.2 20250303) (FINAL)_Legal15102025.docx`

Outputs a single combined CSV with both comments and revisions.

In [ ]:
import win32com.client
import os
import pandas as pd

DOC_PATH = os.path.abspath(
    "docs/KKP/LNC/REVISEDMutual Nondisclosure Agreement (V2.2 20250303) (FINAL)_Legal15102025.docx"
)
print(f"Document: {DOC_PATH}")
print(f"Exists: {os.path.exists(DOC_PATH)}")

In [ ]:
# Word revision type constants
# https://learn.microsoft.com/en-us/office/vba/api/word.wdrevisiontype
REVISION_TYPES = {
    0: "NoRevision",
    1: "Insertion",
    2: "Deletion",
    3: "Property",
    4: "ParagraphNumber",
    5: "DisplayField",
    6: "Reconcile",
    7: "Conflict",
    8: "Style",
    9: "Replace",
    10: "ParagraphProperty",
    11: "TableProperty",
    12: "SectionProperty",
    13: "StyleDefinition",
    14: "MovedFrom",
    15: "MovedTo",
    16: "CellInsertion",
    17: "CellDeletion",
    18: "CellMerge",
    19: "CellSplit",
}

In [ ]:
# Open Word and load the document
word = win32com.client.Dispatch("Word.Application")
word.Visible = False
doc = word.Documents.Open(DOC_PATH)
print(f"Opened: {doc.Name}")
print(f"Pages: {doc.ComputeStatistics(2)}")
print(f"Comments: {doc.Comments.Count}")
print(f"Revisions: {doc.Revisions.Count}")

## Extract Comments & Revisions

In [ ]:
rows = []

# --- Comments ---
for i in range(1, doc.Comments.Count + 1):
    c = doc.Comments(i)
    rows.append({
        "category": "Comment",
        "type": "Comment",
        "author": c.Author,
        "date": str(c.Date),
        "text": c.Range.Text,
        "context": c.Scope.Text,
    })

# --- Track Changes (Revisions) ---
for i in range(1, doc.Revisions.Count + 1):
    r = doc.Revisions(i)
    rev_type = REVISION_TYPES.get(r.Type, f"Unknown({r.Type})")
    rows.append({
        "category": "TrackChange",
        "type": rev_type,
        "author": r.Author,
        "date": str(r.Date),
        "text": r.Range.Text,
        "context": "",
    })

df = pd.DataFrame(rows)
print(f"Total rows: {len(df)}  (Comments: {(df['category']=='Comment').sum()}, TrackChanges: {(df['category']=='TrackChange').sum()})")
print(f"\nType breakdown:")
print(df["type"].value_counts().to_string())
display(df)

In [ ]:
output_path = "docs/KKP/LNC/nda_comments_and_changes.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"Saved {len(df)} rows to: {output_path}")

In [ ]:
# Close the document and Word
doc.Close(False)
word.Quit()
print("Word closed.")